# ALM — Enhanced MAD Multi-Label Training

This notebook trains the **Enhanced ALM MAD Encoder** for multi-label classification.

- Dataset: Pre-generated multi-label MAD (`data/multilabel_mad/`)
- Task: multi-label classification, **7 classes** (multi-hot labels)
- Audio duration: **10 seconds**
- Architecture: `EnhancedALMEncoder` from `utils/enhanced_alm_mad_encoder.py`

This notebook is **self-contained** and can be run end-to-end from a fresh kernel, provided
`data/multilabel_mad/` exists (generated by `script/generate_multilable_dataset.py`).


## 1. Imports & Setup

In [ ]:
import os
import time
import random
import warnings
import json
from pathlib import Path
import sys

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import librosa
import librosa.display
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    average_precision_score,   # NEW for mAP
    precision_recall_fscore_support,
    f1_score,
    hamming_loss,
)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Imports OK")


## 2. Reproducibility & Device

In [ ]:
# ── Reproducibility ────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED) -> None:
    '''Seed all relevant RNGs for reproducible results.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ── Device ──────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("ALM — Enhanced MAD Multi-Label Training")
print("=" * 60)
print(f"Device        : {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version  : {torch.version.cuda}")
print("=" * 60)


## 3. Configuration

In [ ]:
class Config:
    '''Central configuration for the Enhanced ALM multi-label training run.'''

    # ── Audio ───────────────────────────────────────────────────────────
    SAMPLE_RATE: int = 16000
    DURATION: float = 10.0
    NUM_SAMPLES: int = 160000

    # ── Mel spectrogram ─────────────────────────────────────────────────
    N_MELS: int = 128
    N_FFT: int = 1024
    HOP_LENGTH: int = 320
    F_MIN: float = 20.0
    F_MAX: float = 8000.0

    # ── Classes ─────────────────────────────────────────────────────────
    NUM_CLASSES: int = 7
    CLASS_NAMES = [
        "Communication",
        "Gunshot",
        "Footsteps",
        "Shelling",
        "Vehicle",
        "Helicopter",
        "Fighter",
    ]

    # ── Model (EnhancedALMEncoder) ──────────────────────────────────────
    EMBED_DIM: int = 256        # was 512 in single-label
    NUM_HEADS: int = 8
    NUM_LAYERS: int = 6
    FF_DIM: int = 1024
    DROPOUT: float = 0.1        # was 0.4 in single-label
    USE_FUSION: bool = True       # ablation flag
    USE_CLASS_QUERIES: bool = True  # ablation flag

    # ── SpecAugment ─────────────────────────────────────────────────────
    FREQ_MASK_PARAM: int = 27
    TIME_MASK_PARAM: int = 40

    # ── Training ────────────────────────────────────────────────────────
    BATCH_SIZE: int = 16          # smaller because the model is bigger
    EPOCHS: int = 60
    LEARNING_RATE: float = 1e-4   # lower for Transformer
    WEIGHT_DECAY: float = 1e-4
    PATIENCE: int = 10
    POS_WEIGHT_CAP: float = 8.0
    NUM_WORKERS: int = 0
    AMP: bool = True

    # ── Threshold optimization ──────────────────────────────────────────
    THRESH_MIN: float = 0.05
    THRESH_MAX: float = 0.95
    THRESH_STEP: float = 0.05


CFG = Config()

print("=" * 60)
print("Configuration")
print("=" * 60)
for key, value in vars(Config).items():
    if not key.startswith("_"):
        print(f"{key:20s}: {value}")
print("=" * 60)


## 4. Project Paths

In [ ]:
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "multilabel_mad"
AUDIO_DIR = DATA_DIR / "audio"
METADATA_FILE = DATA_DIR / "metadata.csv"
CONFIG_SNAPSHOT_FILE = DATA_DIR / "config.json"

TEST_DATA_DIR = PROJECT_ROOT / "data" / "processed_mad"
TEST_METADATA_FILE = TEST_DATA_DIR / "metadata.csv"

MODEL_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

BEST_MODEL_PATH = MODEL_DIR / "enhanced_alm_mad_best.pth"
FINAL_MODEL_PATH = MODEL_DIR / "enhanced_alm_mad_final.pth"
THRESHOLDS_PATH = MODEL_DIR / "thresholds.json"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Project Paths")
print("=" * 60)
print(f"PROJECT_ROOT       : {PROJECT_ROOT}")
print(f"DATA_DIR            : {DATA_DIR}")
print(f"AUDIO_DIR            : {AUDIO_DIR}")
print(f"METADATA_FILE       : {METADATA_FILE}")
print(f"TEST_METADATA_FILE  : {TEST_METADATA_FILE}")
print(f"MODEL_DIR           : {MODEL_DIR}")
print(f"FIGURES_DIR         : {FIGURES_DIR}")
print("=" * 60)

if not METADATA_FILE.exists():
    raise FileNotFoundError(
        f"Multi-label metadata not found at {METADATA_FILE}.\n"
        "Run `script/generate_multilable_dataset.py` first to generate "
        "`data/multilabel_mad/`."
    )

if not AUDIO_DIR.exists():
    raise FileNotFoundError(f"Audio directory not found at {AUDIO_DIR}.")

print("All required paths found.")


## 5. Load & Inspect Metadata

In [ ]:
metadata = pd.read_csv(METADATA_FILE)

print("=" * 60)
print("Metadata Overview")
print("=" * 60)
print(f"Shape   : {metadata.shape}")
print(f"Columns : {list(metadata.columns)}")
print("=" * 60)
metadata.head()


In [ ]:
# ── Split distribution ──────────────────────────────────────────────────
print("Split distribution:")
print(metadata["split"].value_counts())
print()

train_df = metadata[metadata["split"] == "train"].reset_index(drop=True)
val_df = metadata[metadata["split"] == "val"].reset_index(drop=True)

print(f"Train samples: {len(train_df)}")
print(f"Val samples  : {len(val_df)}")


In [ ]:
# ── Mix-type distribution ───────────────────────────────────────────────
print("Mix-type distribution (all splits):")
print(metadata["mix_type"].value_counts())
print()
print("Mix-type distribution (%):")
print((metadata["mix_type"].value_counts(normalize=True) * 100).round(2))


In [ ]:
# ── Per-class label frequency (label_names is comma-separated) ─────────
from collections import Counter

label_counter = Counter()
for names in metadata["label_names"]:
    for name in str(names).split(","):
        label_counter[name.strip()] += 1

class_dist = pd.Series(label_counter).reindex(CFG.CLASS_NAMES).fillna(0).astype(int)
print("Class frequency (across all mixtures, all splits):")
print(class_dist)

fig, ax = plt.subplots(figsize=(8, 4))
class_dist.plot(kind="bar", ax=ax, color="#4C9AFF")
ax.set_title("Class Frequency — Multi-Label MAD")
ax.set_ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 6. Compute `pos_weight`

For each class, `pos_weight[c] = min(negative_count[c] / positive_count[c], POS_WEIGHT_CAP)`,
computed from the multi-hot `label_vector` column of the **training** split only.


In [ ]:
def parse_label_vector(s: str) -> np.ndarray:
    '''Parse a comma-separated multi-hot string like '0,1,0,0,0,1,0' into an array.'''
    return np.array([int(x) for x in str(s).split(",")], dtype=np.float32)


train_label_matrix = np.stack(train_df["label_vector"].apply(parse_label_vector).to_numpy())
val_label_matrix = np.stack(val_df["label_vector"].apply(parse_label_vector).to_numpy())

assert train_label_matrix.shape[1] == CFG.NUM_CLASSES, "label_vector width mismatch"

pos_counts = train_label_matrix.sum(axis=0)
neg_counts = train_label_matrix.shape[0] - pos_counts

pos_weight_values = np.minimum(neg_counts / np.maximum(pos_counts, 1), CFG.POS_WEIGHT_CAP)
pos_weight_tensor = torch.tensor(pos_weight_values, dtype=torch.float32)

print("=" * 60)
print("pos_weight per class (train split)")
print("=" * 60)
pw_table = pd.DataFrame({
    "class": CFG.CLASS_NAMES,
    "positives": pos_counts.astype(int),
    "negatives": neg_counts.astype(int),
    "pos_weight": pos_weight_values.round(3),
})
print(pw_table.to_string(index=False))
print("=" * 60)
print(f"pos_weight_tensor: {pos_weight_tensor}")


## 7. Audio Loading & Preprocessing

In [ ]:
def load_audio(path: Path, sr: int = CFG.SAMPLE_RATE) -> np.ndarray:
    '''Load an audio file as mono float32 at the target sample rate.'''
    audio, orig_sr = sf.read(str(path), dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if orig_sr != sr:
        audio = librosa.resample(audio, orig_sr=orig_sr, target_sr=sr)
    return audio


def pad_or_trim(audio: np.ndarray, num_samples: int = CFG.NUM_SAMPLES) -> np.ndarray:
    '''Pad with zeros or trim so the waveform is exactly `num_samples` long.'''
    if len(audio) > num_samples:
        audio = audio[:num_samples]
    elif len(audio) < num_samples:
        audio = np.pad(audio, (0, num_samples - len(audio)))
    return audio


mel_transform = T.MelSpectrogram(
    sample_rate=CFG.SAMPLE_RATE,
    n_fft=CFG.N_FFT,
    hop_length=CFG.HOP_LENGTH,
    n_mels=CFG.N_MELS,
    f_min=CFG.F_MIN,
    f_max=CFG.F_MAX,
)
amplitude_to_db = T.AmplitudeToDB()


def preprocess_audio(path: Path) -> torch.Tensor:
    '''Load, pad/trim, and convert a waveform to a log-mel spectrogram tensor (1, n_mels, T).'''
    audio = load_audio(path)
    audio = pad_or_trim(audio)
    waveform = torch.from_numpy(audio).unsqueeze(0)
    mel = mel_transform(waveform)
    log_mel = amplitude_to_db(mel)
    return log_mel


print("Audio preprocessing functions defined.")


In [ ]:
# ── Sanity check on a real file ─────────────────────────────────────────
sample_row = train_df.iloc[0]
sample_path = AUDIO_DIR / sample_row["file"]

waveform = load_audio(sample_path)
waveform = pad_or_trim(waveform)
log_mel_sample = preprocess_audio(sample_path)

print(f"File           : {sample_row['file']}")
print(f"Labels         : {sample_row['label_names']}")
print(f"Mix type       : {sample_row['mix_type']}")
print(f"Waveform shape : {waveform.shape}")
print(f"Log-mel shape  : {tuple(log_mel_sample.shape)}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(waveform)
axes[0].set_title(f"Waveform — {sample_row['file']}")
axes[0].set_xlabel("Sample")

img = librosa.display.specshow(
    log_mel_sample.squeeze(0).numpy(),
    sr=CFG.SAMPLE_RATE,
    hop_length=CFG.HOP_LENGTH,
    x_axis="time",
    y_axis="mel",
    ax=axes[1],
)
axes[1].set_title(f"Log-Mel Spectrogram — {sample_row['label_names']}")
fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
plt.tight_layout()
plt.show()


## 8. Dataset Class

`MADMultiLabelDataset` returns `{"mel": tensor, "label": tensor}` where `label` is a
**float multi-hot vector** `(7,)`, lazily loading and preprocessing audio on the fly.


In [ ]:
class MADMultiLabelDataset(Dataset):
    '''Lazy-loading multi-label MAD dataset built from the pre-generated mixtures.'''

    def __init__(
        self,
        dataframe: pd.DataFrame,
        audio_dir: Path,
        augment: bool = False,
        mel_mean: float = 0.0,
        mel_std: float = 1.0,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.augment = augment
        self.mel_mean = mel_mean
        self.mel_std = mel_std

        if self.augment:
            self.freq_mask = T.FrequencyMasking(freq_mask_param=CFG.FREQ_MASK_PARAM)
            self.time_mask = T.TimeMasking(time_mask_param=CFG.TIME_MASK_PARAM)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        path = self.audio_dir / row["file"]

        log_mel = preprocess_audio(path)
        log_mel = (log_mel - self.mel_mean) / (self.mel_std + 1e-6)

        if self.augment:
            log_mel = self.freq_mask(log_mel)
            log_mel = self.time_mask(log_mel)

        label_vec = parse_label_vector(row["label_vector"])
        label = torch.tensor(label_vec, dtype=torch.float32)

        return {"mel": log_mel, "label": label}


print("MADMultiLabelDataset defined.")


## 9. DataLoaders

In [ ]:
def compute_global_mel_stats(dataframe: pd.DataFrame, audio_dir: Path, n_samples: int = 200) -> tuple[float, float]:
    '''Estimate global mean/std of log-mel values from a random subset of the training data.'''
    rng = np.random.RandomState(SEED)
    sample_idx = rng.choice(len(dataframe), size=min(n_samples, len(dataframe)), replace=False)

    values = []
    for idx in tqdm(sample_idx, desc="Computing global mel stats"):
        row = dataframe.iloc[idx]
        path = audio_dir / row["file"]
        log_mel = preprocess_audio(path)
        values.append(log_mel.numpy().ravel())

    all_values = np.concatenate(values)
    return float(all_values.mean()), float(all_values.std())


MEL_MEAN, MEL_STD = compute_global_mel_stats(train_df, AUDIO_DIR)
print(f"Global mel mean: {MEL_MEAN:.4f}")
print(f"Global mel std : {MEL_STD:.4f}")


In [ ]:
train_dataset = MADMultiLabelDataset(train_df, AUDIO_DIR, augment=True, mel_mean=MEL_MEAN, mel_std=MEL_STD)
val_dataset = MADMultiLabelDataset(val_df, AUDIO_DIR, augment=False, mel_mean=MEL_MEAN, mel_std=MEL_STD)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    num_workers=CFG.NUM_WORKERS,
    pin_memory=True,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")


In [ ]:
# ── Batch sanity check ──────────────────────────────────────────────────
batch = next(iter(train_loader))
print(f"mel batch shape  : {batch['mel'].shape}")
print(f"label batch shape: {batch['label'].shape}")
print(f"label dtype      : {batch['label'].dtype}")
print(f"Example label row: {batch['label'][0]}")


## 10. Model Creation

In [ ]:
from utils import EnhancedALMEncoder

model = EnhancedALMEncoder(
    embed_dim=CFG.EMBED_DIM,
    num_classes=CFG.NUM_CLASSES,
    num_heads=CFG.NUM_HEADS,
    num_layers=CFG.NUM_LAYERS,
    ff_dim=CFG.FF_DIM,
    dropout=CFG.DROPOUT,
    use_fusion=CFG.USE_FUSION,
    use_class_queries=CFG.USE_CLASS_QUERIES,
).to(device)

n_params = model.count_params()

print("=" * 60)
print("EnhancedALMEncoder")
print("=" * 60)
print(f"embed_dim         : {CFG.EMBED_DIM}")
print(f"num_classes       : {CFG.NUM_CLASSES}")
print(f"num_heads         : {CFG.NUM_HEADS}")
print(f"num_layers        : {CFG.NUM_LAYERS}")
print(f"ff_dim            : {CFG.FF_DIM}")
print(f"dropout           : {CFG.DROPOUT}")
print(f"use_fusion        : {CFG.USE_FUSION}")
print(f"use_class_queries : {CFG.USE_CLASS_QUERIES}")
print("-" * 60)
print(f"Total parameters  : {n_params:,}")
print("=" * 60)


In [ ]:
# ── Forward-pass sanity check ────────────────────────────────────────────
with torch.no_grad():
    dummy_mel = batch["mel"].to(device)
    dummy_logits = model(dummy_mel)
    dummy_embed = model.encode(dummy_mel)

print(f"Input mel shape    : {tuple(dummy_mel.shape)}")
print(f"Output logits shape: {tuple(dummy_logits.shape)}  (raw, no sigmoid)")
print(f"Embedding shape    : {tuple(dummy_embed.shape)}")
print(f"Embedding L2 norms : {dummy_embed.norm(dim=1)[:5].cpu().numpy().round(4)}")


## 11. Loss, Optimizer & Scheduler

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor.to(device))

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.LEARNING_RATE,
    weight_decay=CFG.WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG.EPOCHS,
)

scaler = torch.amp.GradScaler("cuda", enabled=(CFG.AMP and torch.cuda.is_available()))

print("Loss     : BCEWithLogitsLoss(pos_weight=...)")
print("Optimizer: AdamW")
print(f"  lr           = {CFG.LEARNING_RATE}")
print(f"  weight_decay = {CFG.WEIGHT_DECAY}")
print("Scheduler: CosineAnnealingLR")
print(f"  T_max = {CFG.EPOCHS}")
print(f"AMP enabled: {scaler.is_enabled()}")


## 12. Training & Evaluation Functions

The primary metric is **mAP**, computed with `average_precision_score(y_true, y_scores, average="macro")`
on the **sigmoid probabilities** (not raw logits).


In [ ]:
def compute_mAP(y_true: np.ndarray, y_scores: np.ndarray) -> float:
    '''Macro mean Average Precision across all classes.'''
    try:
        return float(average_precision_score(y_true, y_scores, average="macro"))
    except ValueError:
        return float("nan")


def train_one_epoch(model, loader, optimizer, criterion, scaler, device, epoch, total_epochs) -> tuple[float, float]:
    model.train()
    running_loss = 0.0
    all_targets, all_scores = [], []

    pbar = tqdm(loader, desc=f"Epoch {epoch:02d}/{total_epochs} [train]", leave=False)
    for batch in pbar:
        mel = batch["mel"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=CFG.AMP):
            logits = model(mel)
            loss = criterion(logits, label)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * mel.size(0)
        all_targets.append(label.detach().cpu().numpy())
        all_scores.append(torch.sigmoid(logits).detach().cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = running_loss / len(loader.dataset)
    y_true = np.concatenate(all_targets)
    y_scores = np.concatenate(all_scores)
    mAP = compute_mAP(y_true, y_scores)

    return avg_loss, mAP


@torch.no_grad()
def evaluate(model, loader, criterion, device, desc="Validation") -> tuple[float, float]:
    model.eval()
    running_loss = 0.0
    all_targets, all_scores = [], []

    pbar = tqdm(loader, desc=desc, leave=False)
    for batch in pbar:
        mel = batch["mel"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=CFG.AMP):
            logits = model(mel)
            loss = criterion(logits, label)

        running_loss += loss.item() * mel.size(0)
        all_targets.append(label.cpu().numpy())
        all_scores.append(torch.sigmoid(logits).cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = running_loss / len(loader.dataset)
    y_true = np.concatenate(all_targets)
    y_scores = np.concatenate(all_scores)
    mAP = compute_mAP(y_true, y_scores)

    return avg_loss, mAP


print("train_one_epoch() and evaluate() defined.")


## 13. Training Loop

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_mAP": [], "val_mAP": [], "lr": []}

best_val_mAP = -1.0
best_epoch = -1
epochs_no_improve = 0

print("=" * 90)
print("Starting Training")
print("=" * 90)

for epoch in range(1, CFG.EPOCHS + 1):
    t0 = time.time()

    train_loss, train_mAP = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, device, epoch, CFG.EPOCHS
    )
    val_loss, val_mAP = evaluate(model, val_loader, criterion, device, desc=f"Epoch {epoch:02d} [val]")

    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]
    elapsed = time.time() - t0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_mAP"].append(train_mAP)
    history["val_mAP"].append(val_mAP)
    history["lr"].append(current_lr)

    improved = val_mAP > best_val_mAP
    marker = ""
    if improved:
        best_val_mAP = val_mAP
        best_epoch = epoch
        epochs_no_improve = 0
        marker = "  ✅ Best model saved"

        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "val_loss": val_loss,
            "val_mAP": val_mAP,
            "config": {k: v for k, v in vars(Config).items() if not k.startswith("_")},
            "label_classes": CFG.CLASS_NAMES,
            "dataset": "MAD",
            "multi_label": True,
            "pos_weight": pos_weight_tensor.tolist(),
        }, BEST_MODEL_PATH)
    else:
        epochs_no_improve += 1

    print(
        f"Epoch {epoch:02d}/{CFG.EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val mAP: {val_mAP:.4f} | LR: {current_lr:.2e} | "
        f"Time: {elapsed:.1f}s{marker}"
    )

    if epochs_no_improve >= CFG.PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch} (no improvement for {CFG.PATIENCE} epochs).")
        break

print("=" * 90)
print(f"Training complete. Best val mAP: {best_val_mAP:.4f} at epoch {best_epoch}.")
print("=" * 90)


## 14. Save Final Model

In [ ]:
torch.save({
    "epoch": len(history["train_loss"]),
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "scheduler_state": scheduler.state_dict(),
    "val_loss": history["val_loss"][-1],
    "val_mAP": history["val_mAP"][-1],
    "history": history,
    "config": {k: v for k, v in vars(Config).items() if not k.startswith("_")},
    "label_classes": CFG.CLASS_NAMES,
    "dataset": "MAD",
    "multi_label": True,
    "pos_weight": pos_weight_tensor.tolist(),
}, FINAL_MODEL_PATH)

print(f"Final model saved to: {FINAL_MODEL_PATH}")
print(f"Best model saved to : {BEST_MODEL_PATH}")


## 15. Training Curves

In [ ]:
plt.style.use("dark_background")

fig = plt.figure(figsize=(14, 5))
gs = gridspec.GridSpec(1, 2, figure=fig)

ax0 = fig.add_subplot(gs[0, 0])
ax0.plot(history["train_loss"], label="Train Loss", color="#4C9AFF")
ax0.plot(history["val_loss"], label="Val Loss", color="#FF5C5C")
ax0.axvline(best_epoch - 1, color="#FFD166", linestyle="--", alpha=0.7, label="Best Epoch")
ax0.set_title("Loss")
ax0.set_xlabel("Epoch")
ax0.set_ylabel("BCE Loss")
ax0.legend()

ax1 = fig.add_subplot(gs[0, 1])
ax1.plot(history["train_mAP"], label="Train mAP", color="#4C9AFF")
ax1.plot(history["val_mAP"], label="Val mAP", color="#FF5C5C")
ax1.axvline(best_epoch - 1, color="#FFD166", linestyle="--", alpha=0.7, label="Best Epoch")
ax1.set_title("Mean Average Precision (mAP)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("mAP")
ax1.legend()

plt.tight_layout()
fig_path = FIGURES_DIR / "enhanced_alm_mad_training_curves.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Training curves saved to: {fig_path}")


## 16. Load Best Model

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

print(f"Loaded best checkpoint from epoch {checkpoint['epoch']}")
print(f"  val_loss: {checkpoint['val_loss']:.4f}")
print(f"  val_mAP : {checkpoint['val_mAP']:.4f}")


## 17. Threshold Optimization

Sweep per-class thresholds from `THRESH_MIN` to `THRESH_MAX` on the validation set and
keep the threshold that maximizes F1 for each class independently.


In [ ]:
@torch.no_grad()
def collect_probs_and_targets(model, loader, device) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    all_targets, all_scores = [], []
    for batch in tqdm(loader, desc="Collecting val predictions", leave=False):
        mel = batch["mel"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)
        logits = model(mel)
        all_targets.append(label.cpu().numpy())
        all_scores.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(all_targets), np.concatenate(all_scores)


val_y_true, val_y_scores = collect_probs_and_targets(model, val_loader, device)

thresholds_grid = np.arange(CFG.THRESH_MIN, CFG.THRESH_MAX + 1e-9, CFG.THRESH_STEP)

best_thresholds = {}
best_f1_per_class = {}

for c, class_name in enumerate(CFG.CLASS_NAMES):
    best_f1 = -1.0
    best_t = 0.5
    for t in thresholds_grid:
        preds = (val_y_scores[:, c] >= t).astype(int)
        f1 = f1_score(val_y_true[:, c], preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = float(t)
    best_thresholds[class_name] = round(best_t, 3)
    best_f1_per_class[class_name] = round(float(best_f1), 4)

thresholds_dict = best_thresholds

print("=" * 60)
print("Optimized Per-Class Thresholds")
print("=" * 60)
thr_table = pd.DataFrame({
    "class": CFG.CLASS_NAMES,
    "threshold": [best_thresholds[c] for c in CFG.CLASS_NAMES],
    "val_F1_at_threshold": [best_f1_per_class[c] for c in CFG.CLASS_NAMES],
})
print(thr_table.to_string(index=False))
print("=" * 60)

with open(THRESHOLDS_PATH, "w") as f:
    json.dump(thresholds_dict, f, indent=2)

print(f"\nThresholds saved to: {THRESHOLDS_PATH}")


## 18. Test Evaluation

The official MAD test set (`data/processed_mad/metadata.csv`, `split == "test"`) is clean,
single-label, and never mixed. Single labels are converted to multi-hot vectors for scoring,
and the optimized per-class thresholds are used to produce binary predictions.


In [ ]:
if not TEST_METADATA_FILE.exists():
    raise FileNotFoundError(f"Official MAD test metadata not found at {TEST_METADATA_FILE}.")

test_full_df = pd.read_csv(TEST_METADATA_FILE)
test_df = test_full_df[test_full_df["split"] == "test"].reset_index(drop=True)
print(f"Test samples (official MAD, clean, single-label): {len(test_df)}")
test_df.head()


In [ ]:
def single_label_to_multihot(label_id: int, num_classes: int = CFG.NUM_CLASSES) -> np.ndarray:
    vec = np.zeros(num_classes, dtype=np.float32)
    vec[int(label_id)] = 1.0
    return vec


# Detect the single-label column name used in the official test metadata.
_label_col_candidates = ["label", "label_id", "class_id", "class"]
_label_col = next((c for c in _label_col_candidates if c in test_df.columns), None)
if _label_col is None:
    raise KeyError(
        f"Could not find a single-label column in test metadata. "
        f"Looked for {_label_col_candidates}, found columns: {list(test_df.columns)}"
    )

print(f"Using '{_label_col}' as the single-label column for the test set.")

TEST_AUDIO_DIR = TEST_DATA_DIR / "audio"
if not TEST_AUDIO_DIR.exists():
    # Fall back to a flat layout if there is no dedicated audio/ subfolder.
    TEST_AUDIO_DIR = TEST_DATA_DIR


class MADTestDataset(Dataset):
    '''Official clean single-label MAD test set, scored as multi-label (one-hot targets).'''

    def __init__(self, dataframe: pd.DataFrame, audio_dir: Path, label_col: str, mel_mean: float, mel_std: float):
        self.df = dataframe.reset_index(drop=True)
        self.audio_dir = Path(audio_dir)
        self.label_col = label_col
        self.mel_mean = mel_mean
        self.mel_std = mel_std

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        path = self.audio_dir / row["file"]

        log_mel = preprocess_audio(path)
        log_mel = (log_mel - self.mel_mean) / (self.mel_std + 1e-6)

        label = torch.tensor(single_label_to_multihot(row[self.label_col]), dtype=torch.float32)
        return {"mel": log_mel, "label": label}


test_dataset = MADTestDataset(test_df, TEST_AUDIO_DIR, _label_col, MEL_MEAN, MEL_STD)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)

print(f"Test batches: {len(test_loader)}")


In [ ]:
test_y_true, test_y_scores = collect_probs_and_targets(model, test_loader, device)

threshold_vec = np.array([thresholds_dict[c] for c in CFG.CLASS_NAMES])
test_y_pred = (test_y_scores >= threshold_vec[None, :]).astype(int)

test_mAP = compute_mAP(test_y_true, test_y_scores)
test_macro_f1 = f1_score(test_y_true, test_y_pred, average="macro", zero_division=0)
test_micro_f1 = f1_score(test_y_true, test_y_pred, average="micro", zero_division=0)
test_hamming = hamming_loss(test_y_true, test_y_pred)

print("=" * 60)
print("Test Set Evaluation (official MAD, thresholds applied)")
print("=" * 60)
print(f"mAP       : {test_mAP:.4f}")
print(f"Macro F1  : {test_macro_f1:.4f}")
print(f"Micro F1  : {test_micro_f1:.4f}")
print(f"Hamming   : {test_hamming:.4f}")
print("=" * 60)


## 19. Per-Class Metrics

In [ ]:
per_class_ap = []
for c in range(CFG.NUM_CLASSES):
    try:
        ap = average_precision_score(test_y_true[:, c], test_y_scores[:, c])
    except ValueError:
        ap = float("nan")
    per_class_ap.append(ap)

precision, recall, f1, support = precision_recall_fscore_support(
    test_y_true, test_y_pred, average=None, zero_division=0
)

per_class_table = pd.DataFrame({
    "class": CFG.CLASS_NAMES,
    "AP": np.round(per_class_ap, 4),
    "precision": np.round(precision, 4),
    "recall": np.round(recall, 4),
    "F1": np.round(f1, 4),
    "support": support.astype(int),
    "threshold": [thresholds_dict[c] for c in CFG.CLASS_NAMES],
})

print("=" * 80)
print("Per-Class Metrics (Test Set)")
print("=" * 80)
print(per_class_table.to_string(index=False))
print("=" * 80)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(CFG.CLASS_NAMES, per_class_ap, color="#4C9AFF")
ax.set_title("Per-Class Average Precision (Test Set)")
ax.set_ylabel("AP")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 20. Embedding Sanity Check

In [ ]:
with torch.no_grad():
    check_batch = next(iter(test_loader))
    check_mel = check_batch["mel"].to(device)
    embeddings = model.encode(check_mel)

norms = embeddings.norm(dim=1).cpu().numpy()

print(f"Embedding shape : {tuple(embeddings.shape)}")
print(f"L2 norms (first 10): {norms[:10].round(4)}")
print(f"Mean norm       : {norms.mean():.4f}")
print(f"Std norm        : {norms.std():.4f}")

assert np.allclose(norms, 1.0, atol=1e-3), "Embeddings are not L2-normalized!"
print("\n✅ Embeddings are L2-normalized as expected.")


## 21. Example Predictions

In [ ]:
n_examples = min(10, len(test_df))
example_idx = np.random.RandomState(SEED).choice(len(test_df), size=n_examples, replace=False)

rows = []
for idx in example_idx:
    row = test_df.iloc[idx]
    true_label = CFG.CLASS_NAMES[int(row[_label_col])]

    probs = test_y_scores[idx]
    preds = test_y_pred[idx]
    predicted_classes = [CFG.CLASS_NAMES[c] for c in range(CFG.NUM_CLASSES) if preds[c] == 1]

    rows.append({
        "file": row["file"],
        "true_label": true_label,
        "predicted": ", ".join(predicted_classes) if predicted_classes else "(none)",
        "top_prob_class": CFG.CLASS_NAMES[int(np.argmax(probs))],
        "top_prob": round(float(np.max(probs)), 4),
        "probs": np.round(probs, 3).tolist(),
    })

examples_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
examples_df


## 22. Final Summary

In [ ]:
W = 58
summary_lines = [
    "╔" + "═" * W + "╗",
    "║" + "  ENHANCED ALM MAD — MULTI-LABEL TRAINING SUMMARY".ljust(W) + "║",
    "╠" + "═" * W + "╣",
    "║" + f" Best epoch          : {best_epoch}".ljust(W) + "║",
    "║" + f" Best val mAP        : {best_val_mAP:.4f}".ljust(W) + "║",
    "║" + f" Test mAP            : {test_mAP:.4f}".ljust(W) + "║",
    "║" + f" Test Macro F1       : {test_macro_f1:.4f}".ljust(W) + "║",
    "║" + f" Test Micro F1       : {test_micro_f1:.4f}".ljust(W) + "║",
    "║" + f" Test Hamming Loss   : {test_hamming:.4f}".ljust(W) + "║",
    "║" + f" Total parameters    : {n_params:,}".ljust(W) + "║",
    "╠" + "═" * W + "╣",
    "║" + f" Best checkpoint     : {BEST_MODEL_PATH.name}".ljust(W) + "║",
    "║" + f" Final checkpoint    : {FINAL_MODEL_PATH.name}".ljust(W) + "║",
    "║" + f" Thresholds file     : {THRESHOLDS_PATH.name}".ljust(W) + "║",
    "╚" + "═" * W + "╝",
]
print("\n".join(summary_lines))
